In [79]:
import json
import pandas as pd

In [80]:
path = 'task1.jsonl'
with open(path, 'r') as file:
    train_data = [json.loads(line) for line in file] 

In [92]:
df = pd.DataFrame(train_data)

In [93]:
df

,doc_id,fact,explanation,statute
0,2013.INSC.228.txt,The facts which are essential to be stated fo...,{'The facts which are essential to be stated f...,[IPC 147]
1,1996.INSC.902.txt,16 Praveen Chandra was the Divisional Forest ...,"{'When A-1 started the engine, A-4, A-6 and A-...",[IPC 147]
2,2003.INSC.675.txt,Law was set in motion by PW-1(Chanakya) on th...,{'Law was set in motion by PW-1(Chanakya) on t...,[IPC 147]
3,2009.INSC.116.txt,The incident leading to the prosecution of th...,{'It was the prosecution case that Mahabir son...,[IPC 147]
4,2002.INSC.528.txt,", the complainant Abdul Jabbar s/o Sheikh Mun...",{'As soon as he reached near the shop he heard...,[IPC 147]
...,...,...,...,...
520,2011.INSC.13.txt,The prosecution case is that the victim Nand...,{'On 1994 at about 00 P.M. when the victim Nan...,[IPC 506]
521,2013.INSC.597.txt,The complainant/respondent no.2 herein (Deep...,{'The complainant/respondent no.2 herein (Deep...,[IPC 147]
522,2013.INSC.754.txt,A.One Sunil (PW.1) lodged a complaint with th...,{'A.One Sunil (PW.1) lodged a complaint with t...,"[IPC 147, IPC 506]"
523,2016.INSC.13.txt,The case of the prosecution is that the appel...,{'On 2010 the appellant sexually violated the ...,"[IPC 376, IPC 506]"


In [97]:
from sklearn.model_selection import train_test_split
X = df['fact']
y = df['statute']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

In [98]:
train_df = pd.DataFrame({
    "fact": X_train,
    "statute": y_train
})

test_df = pd.DataFrame({
    "fact": X_test,
    "statute": y_test
})

In [103]:
import numpy as np
import xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, hamming_loss, average_precision_score, classification_report
)
from sklearn.metrics import precision_recall_curve

In [60]:
RNG = 42
EPS = 1e-7  # numerical safety margin, keeps p away from exactly 0 or 1

In [61]:
def softprob(x: np.ndarray) -> np.ndarray:
   return 0.5 * (1.0 + x / (1.0 + np.abs(x)))

In [62]:
def softprob_grad_hess(y_true: np.ndarray, raw_pred: np.ndarray):
   
    x = raw_pred
    p = np.clip(softprob(x), EPS, 1.0 - EPS)
    dpdx = 0.5 / (1.0 + np.abs(x)) ** 2

    grad = (p - y_true) / (p * (1.0 - p)) * dpdx
    hess = (dpdx ** 2) / (p * (1.0 - p))
    hess = np.clip(hess, 1e-6, None)  # keep strictly positive for stability

    return grad, hess

In [63]:
def softprob_objective(y_true: np.ndarray, raw_pred: np.ndarray):
    return softprob_grad_hess(y_true, raw_pred)

In [64]:

# ---------------------------------------------------------------------------
# Multi-label trainer: one booster per label (binary relevance)
# ---------------------------------------------------------------------------
 
class MultiLabelXGBSoftProb:
    def __init__(self, n_labels: int, use_softprob: bool = True,
                 auto_balance: bool = True, **xgb_params):
        self.n_labels = n_labels
        self.use_softprob = use_softprob
        self.auto_balance = auto_balance
        self.xgb_params = xgb_params
        self.models = []
        self.thresholds = None  
 
    def fit(self, X, Y, eval_set=None):
        self.models = []
        for i in range(self.n_labels):
            y_i = Y[:, i]
            params = dict(self.xgb_params)
 
            if self.auto_balance and not self.use_softprob:
               
                n_pos = max(int(y_i.sum()), 1)
                n_neg = len(y_i) - n_pos
                params["scale_pos_weight"] = n_neg / n_pos
 
            if self.use_softprob:
               
                model = xgb.XGBClassifier(
                    objective=softprob_objective,
                    eval_metric="logloss",
                    base_score=0.0,
                    **params,
                )
 
            else:
                # standard baseline: sigmoid link via binary:logistic
                model = xgb.XGBClassifier(
                    objective="binary:logistic",
                    eval_metric="logloss",
                    **params,
                )
 
            model.fit(X, y_i)
            self.models.append(model)
        return self
 
    def predict_proba(self, X):
        proba = np.zeros((X.shape[0], self.n_labels))
        for i, model in enumerate(self.models):
            if self.use_softprob:
                # custom objective -> booster output is the raw margin;
                # apply our own link to get a probability.
                raw = model.predict(X, output_margin=True)
                proba[:, i] = softprob(raw)
            else:
                proba[:, i] = model.predict_proba(X)[:, 1]
        return proba
 
    def calibrate_thresholds(self, X_val, Y_val):
        
        proba_val = self.predict_proba(X_val)
        thresholds = np.full(self.n_labels, 0.5)
 
        for i in range(self.n_labels):
            y_true = Y_val[:, i]
            # can't tune meaningfully if the validation split has no
            # positives (or no negatives) for this label -- keep default
            if y_true.sum() == 0 or y_true.sum() == len(y_true):
                continue
            precision, recall, thresh = precision_recall_curve(y_true, proba_val[:, i])
            f1s = 2 * precision * recall / (precision + recall + 1e-9)
            if len(thresh) > 0:
                thresholds[i] = thresh[np.argmax(f1s[:-1])]
 
        self.thresholds = thresholds
        return thresholds
 
    def predict(self, X, threshold=None, ensure_at_least_one: bool = True):
        
        proba = self.predict_proba(X)
 
        if threshold is None:
            thresholds = self.thresholds if self.thresholds is not None else np.full(self.n_labels, 0.5)
        elif np.isscalar(threshold):
            thresholds = np.full(self.n_labels, threshold)
        else:
            thresholds = np.asarray(threshold)
 
        preds = (proba >= thresholds).astype(int)
 
        if ensure_at_least_one:
            empty_rows = preds.sum(axis=1) == 0
            if empty_rows.any():
                top_label_idx = proba[empty_rows].argmax(axis=1)
                preds[np.where(empty_rows)[0], top_label_idx] = 1
 
        return preds
 
    # -----------------------------------------------------------------
    # Explainability: for a predicted label, find which sentence in the
    # source text most likely triggered it.
    # -----------------------------------------------------------------
 
    @staticmethod
    def _patch_xgb_base_score_for_shap(model):
      
        import json
        try:
            booster = model.get_booster()
            config = json.loads(booster.save_config())
            lmp = config.get("learner", {}).get("learner_model_param", {})
            raw = lmp.get("base_score")
            if isinstance(raw, str) and raw.strip().startswith("["):
                cleaned = raw.strip().strip("[]").split(",")[0]
                lmp["base_score"] = str(float(cleaned))
                booster.load_config(json.dumps(config))
        except Exception:
            pass  # best-effort only; let shap raise its own error if this didn't help
        return model
 
    def _feature_contributions(self, label_idx: int, doc_vec):
      
        model = self.models[label_idx]
 
        if getattr(self, "_shap_broken", False):
            return self._fallback_contributions(model, doc_vec)
 
        try:
            import shap
        except ImportError:
            self._shap_broken = True
            return self._fallback_contributions(model, doc_vec)
 
        if not hasattr(self, "_shap_explainers"):
            self._shap_explainers = {}
 
        if label_idx not in self._shap_explainers:
            self._patch_xgb_base_score_for_shap(model)
            try:
                self._shap_explainers[label_idx] = shap.TreeExplainer(model)
            except Exception as e:
                import warnings
                warnings.warn(
                    f"shap.TreeExplainer() failed ({type(e).__name__}: {e}). "
                    "Falling back to feature_importance*tfidf for all explanations "
                    "in this run. This is usually a shap/xgboost version mismatch "
                    "(xgboost>=3.0 changed base_score serialization) -- try "
                    "`pip install --upgrade shap` or `pip install \"xgboost<3.0\"`.",
                    RuntimeWarning,
                )
                self._shap_broken = True
                return self._fallback_contributions(model, doc_vec)
 
        explainer = self._shap_explainers[label_idx]
        try:
            shap_vals = explainer.shap_values(doc_vec)
        except Exception as e:
            import warnings
            warnings.warn(
                f"shap_values() failed ({type(e).__name__}: {e}); falling back "
                "to feature_importance*tfidf for the rest of this run.",
                RuntimeWarning,
            )
            self._shap_broken = True
            return self._fallback_contributions(model, doc_vec)
 
        # TreeExplainer can return a list (per-class) or array depending
        # on version/objective; normalize to a single (n_features,) row.
        if isinstance(shap_vals, list):
            shap_vals = shap_vals[-1]
        contrib = np.asarray(shap_vals)[0]
        return contrib, "shap"
 
    def _fallback_contributions(self, model, doc_vec):
        dense = np.asarray(doc_vec.todense())[0]
        importances = model.feature_importances_
        contrib = dense * importances
        return contrib, "fallback"
 
    def explain(self, text: str, vectorizer, label_names,
                threshold=None, ensure_at_least_one: bool = True,
                max_sentences: int = 5, min_score_ratio: float = 0.4, top_n_terms: int = 5):
       
        import re
 
        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]
        if not sentences:
            sentences = [text.strip()]
 
        doc_vec = vectorizer.transform([text])
        proba = self.predict_proba(doc_vec)[0]
 
        if threshold is None:
            thresholds = self.thresholds if self.thresholds is not None else np.full(self.n_labels, 0.5)
        elif np.isscalar(threshold):
            thresholds = np.full(self.n_labels, threshold)
        else:
            thresholds = np.asarray(threshold)
 
        predicted_idx = [i for i, p in enumerate(proba) if p >= thresholds[i]]
        if not predicted_idx and ensure_at_least_one:
            predicted_idx = [int(np.argmax(proba))]
 
        predicted = [label_names[i] for i in predicted_idx]
 
        analyzer = vectorizer.build_analyzer()
        vocab = vectorizer.vocabulary_
 
        explanations = {}
        for label in predicted:
            idx = label_names.index(label)
            contrib, _method = self._feature_contributions(idx, doc_vec)
 
            scored_sentences = []
            for sent in sentences:
                terms_in_sent = set(analyzer(sent))
                term_scores = []
                for term in terms_in_sent:
                    fidx = vocab.get(term)
                    if fidx is not None and contrib[fidx] > 0:
                        term_scores.append((term, float(contrib[fidx])))
                term_scores.sort(key=lambda t: -t[1])
                sent_score = sum(s for _, s in term_scores)
                scored_sentences.append({
                    "sentence": sent,
                    "score": sent_score,
                    "top_terms": term_scores[:top_n_terms],
                })
 
            scored_sentences.sort(key=lambda s: -s["score"])
 
           
            selected = []
            if scored_sentences and scored_sentences[0]["score"] > 0:
                top_score = scored_sentences[0]["score"]
                for s in scored_sentences:
                    if len(selected) >= max_sentences:
                        break
                    if s["score"] <= 0 or s["score"] < min_score_ratio * top_score:
                        break
                    selected.append(s)
 
            explanations[label] = selected
 
        return {
            "predicted_labels": predicted,
            "explanations": explanations,
            "probabilities": {label_names[i]: float(proba[i]) for i in range(self.n_labels)},
            "thresholds_used": {label_names[i]: float(thresholds[i]) for i in range(self.n_labels)},
        }
 

In [65]:
import ast
 
def parse_label_cell(cell, label_sep=","):
   
    s = str(cell).strip()
 
    # try treating it as an actual Python literal first, e.g. "['A', 'B']"
    try:
        val = ast.literal_eval(s)
        if isinstance(val, (list, tuple, set)):
            return [str(x).strip().strip("'\"") for x in val if str(x).strip()]
    except (ValueError, SyntaxError):
        pass
 
    # fallback: plain delimited string, strip any stray brackets/quotes
    s = s.strip("[]")
    return [tok.strip().strip("'\"") for tok in s.split(label_sep) if tok.strip().strip("'\"")]

In [66]:
def labels_to_matrix(labels, label_names):
    Y = np.zeros((len(labels), len(label_names)), dtype=int)
    for i, doc_labels in enumerate(labels):
        for lab in doc_labels:
            Y[i, label_names.index(lab)] = 1
    return Y

In [67]:
def labels_to_matrix_safe(labels, label_names, source_name="data"):
   
    Y = np.zeros((len(labels), len(label_names)), dtype=int)
    unseen = set()
    for i, doc_labels in enumerate(labels):
        for lab in doc_labels:
            if lab in label_names:
                Y[i, label_names.index(lab)] = 1
            else:
                unseen.add(lab)
    if unseen:
        print(
            f"WARNING: {len(unseen)} label(s) found in {source_name} but not seen during "
            f"training (dropped from evaluation, since the model can't predict them): {sorted(unseen)}"
        )
    return Y

In [68]:
def matrix_to_labels(Y, label_names):
   
    return [[label_names[i] for i, v in enumerate(row) if v == 1] for row in Y]

In [72]:
def explain_test_set(model, texts, true_labels, vectorizer, label_names,
                      threshold: float = 0.5, max_sentences: int = 5,
                      min_score_ratio: float = 0.4, top_n_terms: int = 5):

    import pandas as pd
 
    rows = []
    for text, true_labs in zip(texts, true_labels):
        result = model.explain(
            text, vectorizer, label_names,
            threshold=threshold, max_sentences=max_sentences,
            min_score_ratio=min_score_ratio, top_n_terms=top_n_terms,
        )
        predicted = result["predicted_labels"]
 
        if not predicted:
            rows.append({
                "text": texts,
                "true_labels": ",".join(true_labs),
                "predicted_label": None,
                "explanation_sentence": None,
                "score": None,
                "top_terms": None,
            })
            continue
 
        for label in predicted:
            for exp in result["explanations"][label]:
                rows.append({
                    "text": text,
                    "true_labels": ",".join(true_labs),
                    "predicted_label": label,
                    "explanation_sentence": exp["sentence"],
                    "score": exp["score"],
                    "top_terms": exp["top_terms"],
                })
 
    return pd.DataFrame(rows)

    import pandas as pd
 
    rows = []
    for text, true_labs in zip(texts, true_labels):
        result = model.explain(
            text, vectorizer, label_names,
            threshold=threshold, max_sentences=max_sentences,
            min_score_ratio=min_score_ratio, top_n_terms=top_n_terms,
        )
        predicted = result["predicted_labels"]
 
        if not predicted:
            rows.append({
                "text": text,
                "true_labels": ",".join(true_labs),
                "predicted_label": None,
                "explanation_sentence": None,
                "score": None,
                "top_terms": None,
            })
            continue
 
        for label in predicted:
            for exp in result["explanations"][label]:
                rows.append({
                    "text": text,
                    "true_labels": ",".join(true_labs),
                    "predicted_label": label,
                    "explanation_sentence": exp["sentence"],
                    "score": exp["score"],
                    "top_terms": exp["top_terms"],
                })
 
    return pd.DataFrame(rows)

In [73]:
def generate_llm_reasoning(client, fact, predicted_labels, explanation_map,
                            model="gpt-4o-mini", provider="openai", max_tokens=400):
    
    if not predicted_labels:
        return "No label crossed the model's decision threshold for this text."

    evidence_lines = [f'- "{sent}" -> flagged for: {", ".join(labs)}'
                       for sent, labs in explanation_map.items()]
    evidence_block = "\n".join(evidence_lines) if evidence_lines else "(no sentence-level evidence extracted)"

    prompt = f"""You are explaining a text classification decision that has ALREADY been made by a trained model. Do not change, add, 
                or second-guess the predicted labels below -- your only job is to explain, in natural language, how the identified 
                sentences support them.
                Text:
                \"\"\"{fact}\"\"\"

                Predicted label(s): {", ".join(predicted_labels)}

                Sentence-level evidence identified by the model:
                {evidence_block}

                Write a short (3-5 sentence) natural-language explanation of how the evidence above supports the predicted label(s). 
                Write in plain prose, not a numbered list or bullet points. Weave the reasoning naturally rather than mechanically 
                restating the label list. Do not introduce any label not already listed above."""

    if provider == "openai":
        response = client.chat.completions.create(
            model=model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content.strip()

    elif provider == "anthropic":
        response = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        return "".join(block.text for block in response.content if block.type == "text").strip()

    else:
        raise ValueError(f"Unknown provider '{provider}'. Use 'openai' or 'anthropic'.")


In [74]:
def _write_records(records, path):
   
    import json
    if str(path).lower().endswith(".jsonl"):
        with open(path, "w", encoding="utf-8") as f:
            for rec in records:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    else:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(records, f, indent=2, ensure_ascii=False)

In [75]:

def generate_explanation_json(model, texts, vectorizer, label_names, output_path,
                               ids=None, id_prefix="ST-PRED",
                               threshold=None, max_sentences: int = 5,
                               min_score_ratio: float = 0.4, top_n_terms: int = 5,
                               llm_client=None, llm_model: str = "gpt-4o-mini",
                               llm_provider: str = "openai"):
   
    import json
    import os
 
    if ids is None:
        ids = [f"{id_prefix}-{i + 1:04d}" for i in range(len(texts))]
    if len(ids) != len(texts):
        raise ValueError(f"ids length ({len(ids)}) must match texts length ({len(texts)})")
 
    output_path = os.path.abspath(output_path)
    out_dir = os.path.dirname(output_path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)  # in case the target directory doesn't exist yet
 
    records = []
    try:
        for i, (doc_id, text) in enumerate(zip(ids, texts)):
            result = model.explain(
                text, vectorizer, label_names,
                threshold=threshold, max_sentences=max_sentences,
                min_score_ratio=min_score_ratio, top_n_terms=top_n_terms,
            )
 
            # sentence -> [labels]; a sentence can serve more than one label if
            # it was a top-contributing sentence for each of them
            explanation_map = {}
            for label, exps in result["explanations"].items():
                for exp in exps:
                    sent = exp["sentence"]
                    explanation_map.setdefault(sent, [])
                    if label not in explanation_map[sent]:
                        explanation_map[sent].append(label)
 
            if llm_client is not None:
                reasoning = generate_llm_reasoning(
                    llm_client, text, result["predicted_labels"], explanation_map,
                    model=llm_model, provider=llm_provider,
                )
            else:
                reasoning = _format_reasoning_trace(label_names, result)
 
            records.append({
                "id": doc_id,
                "fact": text,
                "reasoning_traces": reasoning,
                "explanation": explanation_map,
            })
 
            if (i + 1) % 10 == 0:
                print(f"  processed {i + 1}/{len(texts)} documents...")
 
    except Exception as e:
        # write whatever succeeded so far instead of losing everything on
        # a late-document failure, then re-raise so you still see the error
        partial_path = os.path.splitext(output_path)[0] + "_PARTIAL" + os.path.splitext(output_path)[1]
        _write_records(records, partial_path)
        print(f"FAILED after {len(records)}/{len(texts)} documents. "
              f"Partial results saved to: {partial_path}")
        raise
 
    _write_records(records, output_path)
 
    # verify it actually landed on disk before declaring success
    if not os.path.exists(output_path):
        raise RuntimeError(f"Write appeared to succeed but file not found at: {output_path}")
 
    size_kb = os.path.getsize(output_path) / 1024
    print(f"Wrote {len(records)} records to {output_path} ({size_kb:.1f} KB)")
    return records
 
 
def run_kfold_cv(texts, labels, label_names=None, n_splits: int = 5, val_size: float = 0.15,
                  xgb_params: dict = None, random_state: int = RNG, max_features: int = 2000):
   
    if label_names is None:
        label_names = sorted({lab for doc in labels for lab in doc})
    print(f"Labels found ({len(label_names)}): {label_names}")
 
    Y = labels_to_matrix(labels, label_names)
    texts_arr = np.array(texts, dtype=object)
 
    if xgb_params is None:
        xgb_params = dict(
            n_estimators=200, max_depth=4, learning_rate=0.1,
            subsample=0.9, colsample_bytree=0.9,
            random_state=random_state, n_jobs=-1,
        )
 
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_metrics = []
 
    print(f"\nRunning {n_splits}-fold cross-validation on {len(texts)} documents...")
    for fold_idx, (trainval_idx, test_idx) in enumerate(kf.split(texts_arr)):
        trainval_texts = texts_arr[trainval_idx].tolist()
        Y_trainval = Y[trainval_idx]
        fold_test_texts = texts_arr[test_idx].tolist()
        Y_fold_test = Y[test_idx]
 
        fold_train_text, fold_val_text, Y_fold_train, Y_fold_val = train_test_split(
            trainval_texts, Y_trainval, test_size=val_size, random_state=random_state
        )
 
        vec = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2), stop_words="english")
        X_fold_train = vec.fit_transform(fold_train_text)
        X_fold_val = vec.transform(fold_val_text)
        X_fold_test = vec.transform(fold_test_texts)
 
        model = MultiLabelXGBSoftProb(n_labels=len(label_names), use_softprob=False, **xgb_params)
        model.fit(X_fold_train, Y_fold_train)
        model.calibrate_thresholds(X_fold_val, Y_fold_val)
 
        proba = model.predict_proba(X_fold_test)
        pred = model.predict(X_fold_test)
 
        metrics = {
            "micro_f1": f1_score(Y_fold_test, pred, average="micro", zero_division=0),
            "macro_f1": f1_score(Y_fold_test, pred, average="macro", zero_division=0),
            "hamming_loss": hamming_loss(Y_fold_test, pred),
        }
        try:
            metrics["avg_precision"] = average_precision_score(Y_fold_test, proba, average="macro")
        except ValueError:
            metrics["avg_precision"] = float("nan")
 
        fold_metrics.append(metrics)
        print(f"  Fold {fold_idx + 1}/{n_splits}: " + ", ".join(f"{k}={v:.4f}" for k, v in metrics.items()))
 
    mean_metrics = {k: float(np.nanmean([m[k] for m in fold_metrics])) for k in fold_metrics[0]}
    std_metrics = {k: float(np.nanstd([m[k] for m in fold_metrics])) for k in fold_metrics[0]}
 
    print("\n" + "=" * 60)
    print(f"{n_splits}-fold CV results (mean +/- std across folds):")
    for k in mean_metrics:
        print(f"  {k}: {mean_metrics[k]:.4f} +/- {std_metrics[k]:.4f}")
 
   .
    print("\nTraining final model on all data for downstream use...")
    final_train_text, final_val_text, Y_final_train, Y_final_val = train_test_split(
        texts_arr.tolist(), Y, test_size=val_size, random_state=random_state
    )
    final_vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2), stop_words="english")
    X_final_train = final_vectorizer.fit_transform(final_train_text)
    X_final_val = final_vectorizer.transform(final_val_text)
 
    final_model = MultiLabelXGBSoftProb(n_labels=len(label_names), use_softprob=False, **xgb_params)
    final_model.fit(X_final_train, Y_final_train)
    thresholds = final_model.calibrate_thresholds(X_final_val, Y_final_val)
    print("Calibrated per-label thresholds for the final model:")
    for lab, t in zip(label_names, thresholds):
        print(f"  {lab}: {t:.3f}")
 
    return {
        "fold_metrics": fold_metrics,
        "mean_metrics": mean_metrics,
        "std_metrics": std_metrics,
        "final_model": final_model,
        "final_vectorizer": final_vectorizer,
        "label_names": label_names,
    }

In [105]:

def main(train_df=None,
         test_data_path: str = None, test_df=None,
         text_col: str = "fact", label_col: str = "statute",
         val_size: float = 0.15):
    # ---- load training data ----
    if train_df is not None:
        train_texts, train_labels = load_dataset_from_dataframe(train_df, text_col=text_col, label_col=label_col)
        print(f"Loaded {len(train_texts)} training documents from provided DataFrame")
    elif train_data_path:
        train_texts, train_labels = load_dataset_from_csv(train_data_path, text_col=text_col, label_col=label_col)
        print(f"Loaded {len(train_texts)} training documents from {train_data_path}")
    else:
        train_texts, train_labels = build_demo_dataset()
        print(f"No train_df/train_data_path given -- using {len(train_texts)}-document synthetic demo data")
 
    # ---- load test data ----
    # test_df/test_data_path may or may not have a ground-truth label
   
    if test_df is not None:
        test_has_labels = label_col in test_df.columns
        test_texts, test_labels = load_dataset_from_dataframe(
            test_df, text_col=text_col, label_col=(label_col if test_has_labels else None)
        )
        print(f"Loaded {len(test_texts)} test documents from provided DataFrame "
              f"(labels {'present -- will report metrics' if test_has_labels else 'NOT present -- inference only, no metrics'})")
    elif test_data_path:
        import pandas as pd
        _peek_cols = (pd.read_excel(test_data_path, nrows=0) if str(test_data_path).lower().endswith((".xlsx", ".xls"))
                      else pd.read_csv(test_data_path, nrows=0)).columns
        test_has_labels = label_col in _peek_cols
        test_texts, test_labels = load_dataset_from_csv(
            test_data_path, text_col=text_col, label_col=(label_col if test_has_labels else None)
        )
        print(f"Loaded {len(test_texts)} test documents from {test_data_path} "
              f"(labels {'present -- will report metrics' if test_has_labels else 'NOT present -- inference only, no metrics'})")
    else:
        raise ValueError(
            "No test data provided. Pass test_df=your_test_dataframe or "
            "test_data_path='path/to/test.csv'."
        )
 
    label_names = sorted({lab for doc in train_labels for lab in doc})
    print(f"Labels found in training data ({len(label_names)}): {label_names}")
 
    Y_train_full = labels_to_matrix(train_labels, label_names)
   
    Y_test = labels_to_matrix_safe(test_labels, label_names, source_name="test data") if test_has_labels else None
 
   
    X_train_text, X_val_text, Y_train, Y_val = train_test_split(
        train_texts, Y_train_full, test_size=val_size, random_state=RNG
    )
    X_test_text = test_texts
    print(f"\nSplit: {len(X_train_text)} train / {len(X_val_text)} validation "
          f"(from train_df) / {len(X_test_text)} test (from test_df)")
 
    vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), stop_words="english")
    X_train = vectorizer.fit_transform(X_train_text)
    X_val = vectorizer.transform(X_val_text)
    X_test = vectorizer.transform(X_test_text)
 
    xgb_params = dict(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RNG,
        n_jobs=-1,
    )
 
    print("\n" + "=" * 60)
    print("Training baseline with standard sigmoid (binary:logistic)")
    print("=" * 60)
    model_sigmoid = MultiLabelXGBSoftProb(
        n_labels=len(label_names), use_softprob=False, **xgb_params
    )
    model_sigmoid.fit(X_train, Y_train)
 
  
    thresholds = model_sigmoid.calibrate_thresholds(X_val, Y_val)
    print("\nCalibrated per-label thresholds (vs flat 0.5 default):")
    for lab, t in zip(label_names, thresholds):
        print(f"  {lab}: {t:.3f}")
 
    proba_sig = model_sigmoid.predict_proba(X_test)
    pred_sig = model_sigmoid.predict(X_test)  # uses calibrated thresholds + at-least-one fallback automatically
 
    if test_has_labels:
        def report(name, y_true, y_pred, y_proba):
            print(f"\n--- {name} ---")
            print("Micro F1:      %.4f" % f1_score(y_true, y_pred, average="micro", zero_division=0))
            print("Macro F1:      %.4f" % f1_score(y_true, y_pred, average="macro", zero_division=0))
            print("Hamming Loss:  %.4f" % hamming_loss(y_true, y_pred))
            try:
                print("Avg Precision: %.4f" % average_precision_score(y_true, y_proba, average="macro"))
            except ValueError:
                pass  # can happen on small sets with a label that's all-0/all-1
            print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
 
        report("Sigmoid (baseline)", Y_test, pred_sig, proba_sig)
    else:
        # no ground truth to score against -- just summarize what got predicted
        n_labels_per_doc = pred_sig.sum(axis=1)
        print(f"\n--- Inference on test set (no ground-truth labels available -- no metrics) ---")
        print(f"Predicted label counts per document: min={n_labels_per_doc.min()}, "
              f"max={n_labels_per_doc.max()}, mean={n_labels_per_doc.mean():.2f}")
        for lab, count in zip(label_names, pred_sig.sum(axis=0)):
            print(f"  {lab}: predicted on {count}/{len(pred_sig)} documents")
 
    return model_sigmoid

In [107]:
model = main(
    train_df=train_df,
    test_df=test_df,
    text_col="fact",
    label_col="statute",
    val_size=0.15
)

Loaded 420 training documents from provided DataFrame
Loaded 105 test documents from provided DataFrame (labels present -- will report metrics)
Labels found in training data (7): ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506']

Split: 357 train / 63 validation (from train_df) / 105 test (from test_df)

Training baseline with standard sigmoid (binary:logistic)

Calibrated per-label thresholds (vs flat 0.5 default):
  IPC 147: 0.065
  IPC 201: 0.322
  IPC 302: 0.247
  IPC 376: 0.770
  IPC 420: 0.399
  IPC 498A: 0.361
  IPC 506: 0.542

--- Sigmoid (baseline) ---
Micro F1:      0.7253
Macro F1:      0.7260
Hamming Loss:  0.1020
Avg Precision: 0.7789
              precision    recall  f1-score   support

     IPC 147       0.54      0.65      0.59        20
     IPC 201       0.57      0.67      0.62        12
     IPC 302       0.57      0.94      0.71        31
     IPC 376       1.00      0.93      0.97        15
     IPC 420       0.75      1.00      0.86 